# Atividade 07 - Mini Projeto

Escolha uma API pública real (pode ser o IBGE, o TSE, ou qualquer outra API aberta e gratuita de sua preferência) e construa um notebook que colete um dataset real a partir dela, aplicando os conceitos da Aula 07.

Requisitos mínimos:

✅ Pelo menos uma requisição GET com requests, com tratamento de status/erro (try/except + raise_for_status()).

- [ ] Paginação, se a API escolhida limitar o tamanho da resposta (se a API não paginar, explique por escrito, no próprio notebook, por que não foi necessária).

- [ ] Salvar o resultado em JSON e CSV, com um bloco de proveniência (fonte, endpoint, data da coleta, licença, se conhecida).

- [ ] Um parágrafo final (em uma célula Markdown) descrevendo: qual API foi usada, quantos registros foram coletados e uma observação sobre os dados.


In [28]:
import json
from datetime import UTC

import pandas as pd
import requests as rq

## Constantes
URL = "https://servicodados.ibge.gov.br/api/v1/localidades/estados/rn/municipios"

## Consumir a API do IBGE Municípios - Exemplo RN
- https://servicodados.ibge.gov.br/api/v1/localidades/estados/rn/municipios

In [ ]:
from datetime import datetime


def get_data(url, timeout=10):
    try:
        response = rq.get(url, timeout=10)
        response.raise_for_status()
        return response
    except rq.exceptions.RequestException as e:
        """
        Imprime informações sobre o erro.
        O type(e).__name__ fornece o tipo de exceção que ocorreu,
        e {e} contém a mensagem de erro detalhada.
        """
        print(f"Falha ao buscar {url}: {type(e).__name__}: {e}")
        raise

## Consumir a API do IBGE Municípios - Exemplo RN
response = get_data(URL)
data = response.json()

coletado_em = datetime.now(UTC).isoformat(timespec='seconds')
n_registros = len(data)

metadados = {
    "status": response.status_code,                      # 200
    "motivo": response.reason,                           # "OK"
    "url_final": response.url,                           # depois de redirects
    "tempo_resposta_s": response.elapsed.total_seconds(),
    "encoding": response.encoding,                       # utf-8
    "content_type": response.headers["Content-Type"],
    "data_servidor": response.headers["Date"],
    "tamanho_bytes": len(response.content),
}

print("METADADOS DA REQUISIÇÃO:")
print(json.dumps(metadados, ensure_ascii=False, indent=4))

print("HEADERS DA REQUISIÇÃO (enviados):")
print(json.dumps(dict(response.request.headers), indent=4))

print("HEADERS DA RESPOSTA (recebidos):")
print(json.dumps(dict(response.headers), indent=4))

print("DADOS COLETADOS:")
print(json.dumps(data, ensure_ascii=False, indent=4))

METADADOS DA REQUISIÇÃO:
{
    "status": 200,
    "motivo": "OK",
    "url_final": "https://servicodados.ibge.gov.br/api/v1/localidades/estados/rn/municipios",
    "tempo_resposta_s": 0.858437,
    "encoding": "utf-8",
    "content_type": "application/json; charset=utf-8",
    "data_servidor": "Wed, 21 Oct 2026 13:12:11 GMT",
    "tamanho_bytes": 76383
}
HEADERS DA REQUISIÇÃO (enviados):
{
    "User-Agent": "python-requests/2.32.4",
    "Accept-Encoding": "gzip, deflate, br, zstd",
    "Accept": "*/*",
    "Connection": "keep-alive"
}
HEADERS DA RESPOSTA (recebidos):
{
    "Date": "Wed, 21 Oct 2026 13:12:11 GMT",
    "Content-Type": "application/json; charset=utf-8",
    "Cache-Control": "max-age=2592000",
    "Content-Encoding": "gzip",
    "Expires": "Fri, 20 Nov 2026 13:12:11 GMT",
    "Vary": "Accept-Encoding",
    "Access-Control-Allow-Headers": "Content-Type",
    "Access-Control-Allow-Methods": "GET, POST, OPTIONS",
    "Access-Control-Allow-Origin": "*",
    "Keep-Alive": "time

## Função para paginar o payload

A resposta da função `get_data(url)`retorna uma lista.
Eu chamo 
```py 
json_data = json.dumps(data, ensure_ascii=False, indent=4)
```
- Onde data recebe o resltado da função get_data(url)
- ensure_ascii=False garante os acentos.


### Paginação
- Depois eu crio uma função de paginação para um Dataframe que é mais simples de implementar.  
- Através do pandas eu chamo crio um Dataframe de data que é um json e aloco na variável df.
- A função se explica por si só nos comentários, mas o print joga o json.dumps no resultado da função onde eu quero imprimir a pagina 2 e que contenha 5 registros por página.

##

In [4]:
df = pd.DataFrame(data)

def paginar(df, pagina=1, tamanho_pagina=10):
    """
    Função para paginar um DataFrame.

    Parâmetros:
    df (pd.DataFrame): O DataFrame a ser paginado.
    pagina (int): O número da página a ser retornada.
    tamanho_pagina (int): O número de registros por página.

    Retorna:
    pd.DataFrame: O DataFrame representando a página solicitada.
    """
    inicio = (pagina - 1) * tamanho_pagina
    return{
        "pagina": pagina,
        "tamanho_pagina": tamanho_pagina,
        "total_registros": len(df),
        "total_paginas": (len(df) + tamanho_pagina - 1) // tamanho_pagina,
        "dados": json.loads(df.iloc[inicio:inicio + tamanho_pagina].to_json(orient="records", force_ascii=False)),
    }

print(json.dumps(paginar(df, pagina=2, tamanho_pagina=5), ensure_ascii=False, indent=4))


{
    "pagina": 2,
    "tamanho_pagina": 5,
    "total_registros": 167,
    "total_paginas": 34,
    "dados": [
        {
            "id": 2400604,
            "nome": "Almino Afonso",
            "microrregiao": {
                "id": 24007,
                "nome": "Umarizal",
                "mesorregiao": {
                    "id": 2401,
                    "nome": "Oeste Potiguar",
                    "UF": {
                        "id": 24,
                        "sigla": "RN",
                        "nome": "Rio Grande do Norte",
                        "regiao": {
                            "id": 2,
                            "sigla": "NE",
                            "nome": "Nordeste"
                        }
                    }
                }
            },
            "regiao-imediata": {
                "id": 240010,
                "nome": "Pau dos Ferros",
                "regiao-intermediaria": {
                    "id": 2403,
                    "nome": "

## Salvar os arquivos em JSON e CSV

Salvar o resultado em JSON e CSV, com um bloco de proveniência (fonte, endpoint, data da coleta, licença, se conhecida).

In [23]:
proveniencia = {
    "fonte": "IBGE — API de Localidades (servicodados.ibge.gov.br)",
    "endpoint": URL,
    "coletado_em": coletado_em,
    "licenca": "Sem licença formal no endpoint; dado público do IBGE, reutilizável com citação da fonte (IBGE)",
    "total_registros": len(data),
    "requisicao": metadados,
}

def salvar_dados(data, proveniencia):
    """
    Salva os dados e a proveniência em arquivos JSON e CSV.

    Parâmetros:
    data (list): Lista de dicionários contendo os dados dos municípios.
    proveniencia (dict): Dicionário contendo informações sobre a proveniência dos dados.
    """
    # Salvar em JSON
    with open("municipios_rn.json", "w", encoding="utf-8") as f:
        json.dump({"proveniencia": proveniencia, "dados": data}, f, ensure_ascii=False, indent=2)

    # Salvar em CSV
    with open("municipios_rn.csv", "w", encoding="utf-8", newline="") as f:
        f.writelines(f"# {chave}: {valor}\n" for chave, valor in proveniencia.items())
        pd.json_normalize(data).to_csv(f, index=False)


salvar_dados(data, proveniencia)

df_csv = pd.read_csv("municipios_rn.csv", comment="#")
assert len(df_csv) == n_registros
df_csv.head()


,id,nome,microrregiao.id,microrregiao.nome,microrregiao.mesorregiao.id,microrregiao.mesorregiao.nome,microrregiao.mesorregiao.UF.id,microrregiao.mesorregiao.UF.sigla,microrregiao.mesorregiao.UF.nome,microrregiao.mesorregiao.UF.regiao.id,...,regiao-imediata.id,regiao-imediata.nome,regiao-imediata.regiao-intermediaria.id,regiao-imediata.regiao-intermediaria.nome,regiao-imediata.regiao-intermediaria.UF.id,regiao-imediata.regiao-intermediaria.UF.sigla,regiao-imediata.regiao-intermediaria.UF.nome,regiao-imediata.regiao-intermediaria.UF.regiao.id,regiao-imediata.regiao-intermediaria.UF.regiao.sigla,regiao-imediata.regiao-intermediaria.UF.regiao.nome
0,2400109,Acari,24012,Seridó Oriental,2402,Central Potiguar,24,RN,Rio Grande do Norte,2,...,240008,Currais Novos,2402,Caicó,24,RN,Rio Grande do Norte,2,NE,Nordeste
1,2400208,Assú,24004,Vale do Açu,2401,Oeste Potiguar,24,RN,Rio Grande do Norte,2,...,240011,Açu,2403,Mossoró,24,RN,Rio Grande do Norte,2,NE,Nordeste
2,2400307,Afonso Bezerra,24009,Angicos,2402,Central Potiguar,24,RN,Rio Grande do Norte,2,...,240011,Açu,2403,Mossoró,24,RN,Rio Grande do Norte,2,NE,Nordeste
3,2400406,Água Nova,24005,Serra de São Miguel,2401,Oeste Potiguar,24,RN,Rio Grande do Norte,2,...,240010,Pau dos Ferros,2403,Mossoró,24,RN,Rio Grande do Norte,2,NE,Nordeste
4,2400505,Alexandria,24006,Pau dos Ferros,2401,Oeste Potiguar,24,RN,Rio Grande do Norte,2,...,240010,Pau dos Ferros,2403,Mossoró,24,RN,Rio Grande do Norte,2,NE,Nordeste


## Baixar os arquivos para entregar

Os arquivos foram salvos pela extensão colab do vscode. Eles não são salvos automaticamente n diretório. A maneira mais fácil de recuperar esses arquivos é fazer o donwload pelo colab.

In [ ]:
from google.colab import files

files.download("municipios_rn.csv")
files.download("municipios_rn.json")

/content/municipios_rn.csv

/content/municipios_rn.json